In [0]:
%sql

SELECT DISTINCT job_title
FROM db_academy.pii_data.employees_raw;

In [0]:
from pyspark.sql.functions import col, lit, when, create_map

# job_title will be grouped into broader categories
job_category_mapping = {
    "Product Manager": "Management",
    "Sales Executive": "Sales & Businesses",
    "Software Engineer": "Engineering",
    "HR Manager": "Management",
    "Data Scientist": "Data & Analytics"
}

mapping_dict_expr = create_map([lit(x) for k, v in job_category_mapping.items() for x in (k, v)])
mapping_expres = when(col("job_title").isin(list(job_category_mapping.keys())), mapping_dict_expr[col("job_title")]).otherwise(lit("Others"))

employees_generialized = (
    spark.read.table("db_academy.pii_data.employees_raw")
    .withColumn("job_categories_mapping", mapping_expres)
    .drop("job_title")
    .withColumnRenamed("job_categories_mapping", "job_title")
)

display(employees_generialized.limit(10))